In [1]:
# 1. Install libraries
!pip install pandas scikit-learn nltk matplotlib

# 2. Import libraries
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

nltk.download('stopwords')
nltk.download('wordnet')

# 3. Load dataset
df = pd.read_csv("Small Cause Court,Pune.csv", encoding="latin1")

print(df.shape)
print(df.columns)

# 4. Select text column
# Change 'Description' to your actual text column name
text_col = "Description"

df[text_col] = df[text_col].fillna("")

# 5. NLP preprocessing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words
             if w not in stop_words and len(w) > 2]
    return " ".join(words)

df["Clean_Text"] = df[text_col].apply(clean_text)

# 6. TF-IDF
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = tfidf.fit_transform(df["Clean_Text"])

# 7. Find best K
scores = {}

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

# 8. K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

# 9. Show cluster results
print(df["Cluster"].value_counts().sort_index())

# 10. PCA visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X.toarray())

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df["Cluster"], cmap="viridis")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clustering - Small Cause Court Pune")
plt.colorbar(label="Cluster")
plt.show()

# 11. Top words in each cluster
terms = tfidf.get_feature_names_out()

for i in range(best_k):
    center = kmeans.cluster_centers_[i]
    words = [terms[j] for j in center.argsort()[-10:][::-1]]
    print("Cluster", i, ":", ", ".join(words))

# 12. Save results
df.to_csv("Small Cause Court,Pune_Clustered.csv", index=False)

print("Done! File saved successfully.")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


(23, 4)
Index(['Sr No', 'Case Type/Case Number/Case Year', 'Order Date', 'Order No.'], dtype='str')


KeyError: 'Description'